# 00 — Kaggle Data Download (MTG-Jamendo)

**Goal:** Pull annotations/splits + log-mel shards into `/kaggle/working/MTG_Instrument`.

## Before you run — turn Internet ON
1. In the Kaggle notebook UI: **Settings** (right sidebar) → **Internet** → **On**
2. Wait a few seconds, then re-run from the top.
3. Without Internet, `git clone` / `wget` fail with `Could not resolve host: github.com` (exactly the error you hit).

**Optional offline path:** Add Data → search/upload `mtg-jamendo-dataset` (or attach a zip of its `data/` folder) under `/kaggle/input/...`, then this notebook will copy annotations without cloning.

**After this notebook:** *Save Version → Save output* and publish as a private dataset so later stages skip re-download.

See `docs/kaggle-pipeline-plan.md`.


In [1]:
# 1) Settings → Internet → On  (required)
# 2) Then run this cell
import socket

def check_internet(host="github.com", port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = check_internet()
print("Internet reachable:", ONLINE)
if not ONLINE:
    print(
        "\n❌ DNS/network blocked. Turn ON Internet in Kaggle notebook Settings,\n"
        "   or attach annotations under /kaggle/input/ and skip the clone cell.\n"
    )
else:
    print("✓ Network OK — safe to download annotations + mel shards")

!pip install -q tqdm


Internet reachable: True
✓ Network OK — safe to download annotations + mel shards


In [2]:
from pathlib import Path
import os, json, random
import numpy as np
import pandas as pd

# Toggle: if you attached a prior Kaggle Dataset output, set True and point INPUT_ROOT
USE_CACHED_INPUT = False
INPUT_ROOT = Path("/kaggle/input/mtg-instrument-cache/MTG_Instrument")  # adjust slug if needed
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")

ROOT = INPUT_ROOT if USE_CACHED_INPUT and INPUT_ROOT.exists() else WORKING_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"

for p in [MEL_DIR, ANN_DIR, FEAT_DIR, CKPT_DIR, RESULTS_DIR, ROOT / "dataset"]:
    if not USE_CACHED_INPUT:
        p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("ROOT =", ROOT)
print("MEL_DIR exists:", MEL_DIR.exists(), "| ANN_DIR exists:", ANN_DIR.exists())


ROOT = /kaggle/working/MTG_Instrument
MEL_DIR exists: True | ANN_DIR exists: True


## 1. Get annotations + official split-0 files

Priority order (no `git clone` required if step A works):
1. Copy from an attached `/kaggle/input/...` dataset
2. `wget` individual raw GitHub files (needs Internet)
3. `git clone` as last resort (needs Internet)


In [3]:
from pathlib import Path
import subprocess, shutil, urllib.request

RAW = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED = [
    # official split-0 genre + instrument subsets
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    # full tag tables (used by later notebooks)
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]

def find_input_data_root() -> Path | None:
    """Prefer an attached Kaggle input that already contains MTG data/."""
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    # common layouts
    candidates = []
    for p in base.rglob("autotagging_genre.tsv"):
        candidates.append(p.parent)
    for p in base.rglob("autotagging_genre-train.tsv"):
        # .../data/splits/split-0/file → data root is parents[2]
        candidates.append(p.parents[2])
    for c in candidates:
        if (c / "autotagging_genre.tsv").exists() or (c / "splits").exists():
            return c
    return None

def copy_tree_annotations(src_data: Path) -> int:
    n = 0
    for rel in NEEDED:
        s = src_data / rel
        if not s.exists():
            # also accept flat copies
            s2 = src_data / Path(rel).name
            s = s2 if s2.exists() else s
        if not s.exists():
            print("missing:", rel)
            continue
        d = ANN_DIR / rel
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
        print("copied", d)
        n += 1
    return n

def wget_annotations() -> int:
    n = 0
    for rel in NEEDED:
        url = f"{RAW}/{rel}"
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        print("wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    return n

src = find_input_data_root()
copied = 0
if src is not None:
    print("Using attached input annotations from:", src)
    copied = copy_tree_annotations(src)
elif check_internet():
    try:
        print("Downloading annotation files via raw.githubusercontent.com (no git clone)...")
        copied = wget_annotations()
    except Exception as e:
        print("wget failed:", e)
        print("Falling back to git clone...")
        REPO = Path("/kaggle/working/mtg-jamendo-dataset")
        if not REPO.exists():
            subprocess.check_call([
                "git", "clone", "--depth", "1",
                "https://github.com/MTG/mtg-jamendo-dataset.git", str(REPO),
            ])
        copied = copy_tree_annotations(REPO / "data")
else:
    raise RuntimeError(
        "No annotations found and Internet is OFF.\n"
        "Fix one of:\n"
        "  A) Kaggle Settings → Internet → On, then re-run this cell\n"
        "  B) Add Data: attach MTG/mtg-jamendo-dataset (or a zip of its data/ folder)"
    )

print(f"\nAnnotation files ready: {copied}")
assert copied > 0, "No annotation files were copied — check Internet / Add Data"
print("Annotation tree:")
for p in sorted(ANN_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ANN_DIR), p.stat().st_size)


wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_genre-train.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_genre-validation.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_genre-test.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_instrument-train.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_instrument-validation.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/splits/split-0/autotagging_instrument-test.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/autotagging_genre.tsv
wget https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data/autotagging_instrument.tsv

Annotation files ready: 8
Annotation tree:
  autotagging_genre.tsv 57

## 2. Download mel-spectrogram shards (Phase 2: 00–02)


In [4]:
import subprocess
from tqdm.auto import tqdm

if not check_internet("cdn.freesound.org"):
    # freesound host check; fall back to generic internet flag
    if not ONLINE:
        raise RuntimeError(
            "Internet is OFF — cannot download mel shards.\n"
            "Turn on Internet, or attach a dataset that already contains extracted .npy mels\n"
            "under MTG_Instrument/dataset/logmel_songs and set USE_CACHED_INPUT=True."
        )

SHARDS = [0, 1, 2]  # expand later: list(range(0, 10))
BASE_URL = "https://cdn.freesound.org/mtg-jamendo/raw_30s/melspecs"

MEL_DIR.mkdir(parents=True, exist_ok=True)

def download_shard(i: int):
    tar_name = f"raw_30s_melspecs-{i:02d}.tar"
    tar_path = MEL_DIR / tar_name
    url = f"{BASE_URL}/{tar_name}"
    marker = MEL_DIR / f".shard_{i:02d}_done"
    if marker.exists():
        print(f"shard {i:02d} already marked done — skip")
        return
    if not tar_path.exists():
        print(f"Downloading {url} ...")
        subprocess.check_call(["wget", "-q", "-O", str(tar_path), url])
    print(f"Extracting {tar_name} ...")
    subprocess.check_call(["tar", "-xf", str(tar_path), "-C", str(MEL_DIR)])
    tar_path.unlink(missing_ok=True)
    marker.write_text("ok")
    print(f"shard {i:02d} ready")

for i in SHARDS:
    download_shard(i)

npy_count = len(list(MEL_DIR.rglob("*.npy")))
print(f"Total .npy files under MEL_DIR: {npy_count}")
assert npy_count > 0, "No mel .npy found — check download / internet"


Extracting raw_30s_melspecs-00.tar ...
shard 00 ready
Extracting raw_30s_melspecs-01.tar ...
shard 01 ready
Extracting raw_30s_melspecs-02.tar ...
shard 02 ready
Total .npy files under MEL_DIR: 1716


## 3. Persist layout summary


In [5]:
summary = {
    "root": str(ROOT),
    "mel_dir": str(MEL_DIR),
    "ann_dir": str(ANN_DIR),
    "n_npy": len(list(MEL_DIR.rglob("*.npy"))),
    "shards": SHARDS,
}
(RESULTS_DIR / "00_download_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nNext: run 01_preprocessing.ipynb (or Save Version → Dataset for caching).")


{
  "root": "/kaggle/working/MTG_Instrument",
  "mel_dir": "/kaggle/working/MTG_Instrument/dataset/logmel_songs",
  "ann_dir": "/kaggle/working/MTG_Instrument/annotations",
  "n_npy": 1716,
  "shards": [
    0,
    1,
    2
  ]
}

Next: run 01_preprocessing.ipynb (or Save Version → Dataset for caching).
